In [1]:
import pandas as pd

train = pd.read_csv("dataset/public_traces.csv")
train["tokens"]

train.head()

,program_id,category,tokens
0,5080,Adware,ldrloaddll ldrgetprocedureaddress ldrgetproced...
1,4607,Adware,ldrloaddll ldrgetprocedureaddress ldrgetproced...
2,2264,Adware,ntallocatevirtualmemory ntfreevirtualmemory nt...
3,3941,Adware,ldrloaddll ldrgetprocedureaddress ldrgetproced...
4,4997,Adware,ldrloaddll ldrgetprocedureaddress ldrgetproced...


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1, 2)
)

X = tfidf.fit_transform(train["tokens"])

print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 893722 stored elements and shape (4843, 9658)>
  Coords	Values
  (0, 4214)	0.05638070171682814
  (0, 3984)	0.44470245344284587
  (0, 6534)	0.008366515554467116
  (0, 2065)	0.004955054290302361
  (0, 8660)	0.007913060269614802
  (0, 1576)	0.016730420210198237
  (0, 8964)	0.005439920234094885
  (0, 1637)	0.008156093395535361
  (0, 4523)	0.007813256033009322
  (0, 5115)	0.1863440011724794
  (0, 6245)	0.04711271410029178
  (0, 6798)	0.02414395525885521
  (0, 3146)	0.03249922149268929
  (0, 5319)	0.0027935446341958905
  (0, 5561)	0.0036629706336025023
  (0, 6084)	0.003432989493228706
  (0, 7940)	0.017472642821731516
  (0, 7244)	0.005274888003474401
  (0, 3891)	0.024158972238256672
  (0, 8051)	0.35987135789461216
  (0, 8286)	0.4263467603214657
  (0, 7554)	0.23804339533235994
  (0, 671)	0.01094513759345074
  (0, 7887)	0.14239356823170737
  (0, 6400)	0.0058074777535028245
  :	:
  (4842, 4395)	0.0005777264772829829
  (4842, 7570)	0.0

In [14]:
from collections import Counter, defaultdict

rows = []
by_origin = defaultdict(list)

for origin, text in enumerate(train["tokens"]):

    tokens = text.split()

    for start in range(0, len(tokens), 150):

        window_tokens = tokens[start:start + 200]

        unigram_counts = sorted(

            Counter(window_tokens).values(),

            reverse=True

        )

        bigrams = zip(window_tokens[:-1], window_tokens[1:])

        bigram_counts = sorted(

            Counter(bigrams).values(),

            reverse=True

        )

        unigram_counts.extend([0] * (200 - len(unigram_counts)))
        bigram_counts.extend([0] * (200 - len(bigram_counts)))

        rows.append({

            "origin": origin,

            "window": " ".join(window_tokens),

            "unigram_counts": unigram_counts,

            "bigram_counts": bigram_counts

        })

for r in rows:
    by_origin[r["origin"]].append(r)

print (rows[:1])

[{'origin': 0, 'window': 'ldrloaddll ldrgetprocedureaddress ldrgetprocedureaddress ldrloaddll ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetprocedureaddress ldrgetpro

In [38]:
import random
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances

pairs = zip(random.sample(rows, 2000), random.sample(rows, 2000))

final_rows = []

for r1, r2 in pairs:
    final_rows.append({
        "same": 1 if r1['origin'] == r2['origin'] else 0,
        "unigram_cosine": cosine_similarity([r1["unigram_counts"]], [r2["unigram_counts"]])[0, 0],
        "bigram_cosine": cosine_similarity([r1["bigram_counts"]], [r2["bigram_counts"]])[0, 0],
        "dist1": pairwise_distances([r1["unigram_counts"]], [r2["unigram_counts"]])[0, 0],
        "dist2": pairwise_distances([r1["bigram_counts"]], [r2["bigram_counts"]])[0, 0]
    })

    candidates = by_origin[r1["origin"]].copy()
    candidates.remove(r1)

    if (len(candidates) == 0):
        continue

    r2 = random.choice(candidates)

    final_rows.append({
        "same": 1 if r1['origin'] == r2['origin'] else 0,
        "unigram_cosine": cosine_similarity([r1["unigram_counts"]], [r2["unigram_counts"]])[0, 0],
        "bigram_cosine": cosine_similarity([r1["bigram_counts"]], [r2["bigram_counts"]])[0, 0],
        "dist1": pairwise_distances([r1["unigram_counts"]], [r2["unigram_counts"]])[0, 0],
        "dist2": pairwise_distances([r1["bigram_counts"]], [r2["bigram_counts"]])[0, 0]
    })

final_df = pd.DataFrame(final_rows)

print(final_df["same"].value_counts())
final_df.head()

same
0    1998
1    1993
Name: count, dtype: int64


,same,unigram_cosine,bigram_cosine,dist1,dist2
0,0,0.999607,0.997175,160.377679,148.892579
1,1,0.999198,0.998537,151.324155,146.311312
2,0,1.000000,1.000000,0.000000,0.000000
3,1,1.000000,1.000000,0.000000,0.000000
4,0,0.603386,0.587675,119.121786,118.970585


In [39]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

rf = RandomForestClassifier(
    n_estimators=350,
    n_jobs=-1,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(final_df.drop(columns=["same"]), final_df["same"], test_size=0.2, random_state=42)

rf.fit(X_train, y_train)
pred = rf.predict_proba(X_test)[:, 1]
score = roc_auc_score(y_test, pred)

score

0.7202725563909775

In [6]:
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torch

class PublicTracesDataset(Dataset):
    def __init__(self, df):
        super().__init__()

        self.df = df

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        row = self.df.iloc[index]
        return row["tokens"]
    
vocab = tfidf.vocabulary_

PAD_ID = len(vocab)
UNK_ID = len(vocab) + 1

def text_to_ids(text):
    return torch.tensor([vocab.get(token, UNK_ID) for token in text.split()], dtype=torch.long)

def collate(batch):
    all_windows = []
    origins = []

    for i, text in enumerate(batch):
        ids = text_to_ids(text)

        if ids.size(0) < 200:
            ids = F.pad(
                ids,
                (0, 200 - ids.size(0)),
                value=PAD_ID
            )

        windows = ids.unfold(0, 200, 150)

        all_windows.append(windows)
        origins.extend([i] * windows.size(0))

    all_windows = torch.cat(all_windows, dim=0)

    return all_windows, torch.tensor(origins)

    
dataset = PublicTracesDataset(train)
loader = DataLoader(dataset, batch_size=8, collate_fn=collate)

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.emb = nn.Embedding(
            len(vocab) + 2,
            32,
            padding_idx=PAD_ID
        )

        self.conv3 = nn.Conv1d(32, 128, 3)
        self.conv5 = nn.Conv1d(32, 128, 5)

        self.pool = nn.AdaptiveMaxPool1d(1)

    def forward(self, x):
        x = self.emb(x)

        x = x.transpose(1, 2)

        act3 = F.relu(self.conv3(x))
        act5 = F.relu(self.conv5(x))

        p3 = self.pool(act3).squeeze(-1)
        p5 = self.pool(act5).squeeze(-1)

        cat = torch.cat([p3, p5], dim=1)

        feat = F.normalize(cat, p=2, dim=1)

        return feat


cnn = CNN().to('mps')

optimizer = torch.optim.AdamW(cnn.parameters(), lr=1e-4)

loss_fn = nn.CosineEmbeddingLoss(margin=0.3)

cnn.train()

for epoch in range(20):
    total_loss = 0

    for windows, origins in loader:
        windows = windows.to("mps")
        origins = origins.to("mps")

        optimizer.zero_grad()

        embeddings = cnn(windows)  # [number_of_windows, 512]

        # Every unique pair: (0,1), (0,2), ..., (n-2,n-1)
        i, j = torch.triu_indices(
            embeddings.size(0),
            embeddings.size(0),
            offset=1,
            device="mps"
        )

        emb1 = embeddings[i]
        emb2 = embeddings[j]

        # +1 if both windows came from the same text, otherwise -1
        targets = torch.where(
            origins[i] == origins[j],
            1.0,
            -1.0
        )

        batch_loss = loss_fn(emb1, emb2, targets)

        batch_loss.backward()
        optimizer.step()

        total_loss += batch_loss.item()

    print(f"Epoch {epoch + 1}: {total_loss / len(loader):.4f}")

Epoch 1: 0.3559


In [43]:
import io, pickle
import numpy as np
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity

class Submission:
    def __init__(self):
        # Load or train your models here. The baseline currently needs nothing.
        pass

    # ===== DO NOT EDIT: the cloud calls this to collect your scores =====
    def __call__(self, data: bytes) -> bytes:
        req = pickle.loads(data)
        fn = self.score_A if req["part"] == "A" else self.score_B
        scores = fn(req["windows"], req["pairs"])
        buf = io.BytesIO(); np.save(buf, np.asarray(list(scores), dtype=np.float64))
        return buf.getvalue()
    # ===================================================================

    def score_A(self, windows, pairs):
        documents = [" ".join(window) for window in windows]

        X = tfidf.transform(documents)

        device = next(cnn.parameters()).device
        cnn.eval()

        ids = torch.stack([
            text_to_ids(doc)
            for doc in documents
        ]).to(device)

        with torch.no_grad():
            embeddings = cnn(ids)

        tfidf_scores = []
        cnn_scores = []

        for i, j in pairs:
            tfidf_scores.append(X[i].multiply(X[j]).sum())

            # embeddings are already L2-normalized
            cnn_scores.append(torch.dot(embeddings[i], embeddings[j]).item())

        tfidf_scores = np.asarray(tfidf_scores, dtype=np.float64)
        cnn_scores = np.asarray(cnn_scores, dtype=np.float64)

        # z-score normalization
        tfidf_scores = (tfidf_scores - tfidf_scores.mean()) / (tfidf_scores.std() + 1e-8)
        cnn_scores = (cnn_scores - cnn_scores.mean()) / (cnn_scores.std() + 1e-8)

        final_scores = tfidf_scores + cnn_scores

        return final_scores

    def score_B(self, windows, pairs):
        def count_profile(window):
            tokens = window.split() if isinstance(window, str) else list(window)
            bigrams = list(zip(tokens[:-1], tokens[1:]))
            return {
                "unigram_counts": sorted(Counter(tokens).values(), reverse=True),
                "bigram_counts": sorted(Counter(bigrams).values(), reverse=True),
            }

        def count_cosine(a, b):
            n = 200
            a.extend([0] * (200 - len(a)))
            b.extend([0] * (200 - len(b)))
            if n == 0:
                return 0.0
            return cosine_similarity([a[:n]], [b[:n]])[0, 0]

        profiles = [count_profile(window) for window in windows]
        features = []

        for i, j in pairs:
            left = profiles[i]
            right = profiles[j]
            features.append({
                "unigram_cosine": count_cosine(left["unigram_counts"], right["unigram_counts"]),
                "bigram_cosine": count_cosine(left["bigram_counts"], right["bigram_counts"]),
                "dist1": pairwise_distances([left["unigram_counts"]], [right["unigram_counts"]])[0, 0],
                "dist2": pairwise_distances([left["bigram_counts"]], [right["bigram_counts"]])[0, 0],
            })

        if not features:
            return []

        X = pd.DataFrame(features, columns=["unigram_cosine", "bigram_cosine", "dist1", "dist2"])
        return rf.predict_proba(X)[:, 1]


In [44]:
cnn.eval()
sol = Submission()   # loads/trains everything

In [45]:
# OPTIONAL local check — self-contained; mirrors the grader (disjoint Part A / Part B
# programs, per-window scramble for B, 50+50 bands). Needs only public_traces.csv.
import csv, random
import numpy as np
from collections import defaultdict
from sklearn.metrics import roc_auc_score

WINDOW = 200; WPP = 8
def _load(p):
    out=[]
    with open(p) as f:
        r=csv.reader(f); next(r)
        for pid,cat,toks in r: out.append({"program_id":int(pid),"category":cat,"tokens":toks.split()})
    return out
def _windows(traces):
    out=[]
    for tr in traces:
        s=tr["tokens"]; n=(len(s)//WINDOW)*WINDOW
        out.extend([{"program_id":tr["program_id"],"category":tr["category"],"wid":j//WINDOW,
                     "tokens":s[j:j+WINDOW]} for j in range(0,n,WINDOW)][:WPP])
    return out
def _pairs(ws,n,seed):
    rng=random.Random(seed); bc=defaultdict(list); bp=defaultdict(list)
    for k,w in enumerate(ws): bc[w["category"]].append(k); bp[w["program_id"]].append(k)
    cats=sorted(bc); multi=[p for p in bp if len(bp[p])>=2]; P=[]; L=[]
    while len(P)<n:
        if rng.random()<0.5:
            p=rng.choice(multi); a,b=rng.sample(bp[p],2); P.append((a,b)); L.append(1)
        else:
            pool=bc[rng.choice(cats)]
            for _ in range(50):
                a,b=rng.sample(pool,2)
                if ws[a]["program_id"]!=ws[b]["program_id"]: P.append((a,b)); L.append(0); break
    return P,L
def _scramble(ws,off):
    vocab=sorted({t for w in ws for t in w["tokens"]}); out=[]
    for w in ws:
        r=random.Random((w["program_id"]*1_000_000+w["wid"])^off); sh=list(vocab); r.shuffle(sh)
        m=dict(zip(vocab,sh)); out.append([m[t] for t in w["tokens"]])
    return out

tr=_load("dataset/public_traces.csv")
by_prog={}
for t in tr: by_prog.setdefault(t["program_id"], t)
ids=sorted(by_prog); random.Random(7).shuffle(ids)
val=ids[:int(len(ids)*0.30)]; h=len(val)//2
wA=_windows([by_prog[i] for i in val[:h]]); WA=[w["tokens"] for w in wA]; pA,lA=_pairs(wA,3000,101)
wB=_windows([by_prog[i] for i in val[h:]]); WB=_scramble(wB,303); pB,lB=_pairs(wB,3000,202)
pts=lambda a,b: max(0.0, min(50.0,(a-0.5)/b*50.0))
aucA=roc_auc_score(lA, sol.score_A(WA,pA)); aucB=roc_auc_score(lB, sol.score_B(WB,pB))
print(f"Part A: AUC {aucA:.3f} -> {pts(aucA,0.34):.1f}/50")
print(f"Part B: AUC {aucB:.3f} -> {pts(aucB,0.28):.1f}/50")
print(f"ESTIMATED TOTAL ~ {pts(aucA,0.34)+pts(aucB,0.28):.1f}/100  (secret set differs slightly)")

Part A: AUC 0.771 -> 39.9/50
Part B: AUC 0.641 -> 25.2/50
ESTIMATED TOTAL ~ 65.0/100  (secret set differs slightly)


In [46]:
# Build submission.pkl  (this is what you upload as the Output)
import cloudpickle, os
# `sol` was trained above. Re-run the train cell first if you restarted the kernel.
with open("submission.pkl", "wb") as f:
    cloudpickle.dump(sol, f)          
mb = os.path.getsize("submission.pkl") / 1e6
print(f"wrote submission.pkl  ({mb:.1f} MB)  -- must be < 50 MB")
assert mb < 50, "too big: cap model size (fewer trees / depth)"

wrote submission.pkl  (35.0 MB)  -- must be < 50 MB
